# QuantumJam 2026 · Curva de energía de $\mathrm{H_4}$ con VQE

Cadena lineal de 4 H, espaciado uniforme $d$, base STO-3G. Enunciado en `docs/`.

## 0. Instalación

In [ ]:
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 pyscf==2.14.0 numpy scipy pandas matplotlib

## 0.1 Contrato compartido

Nombres, firmas y convenciones que usa todo el notebook. Cada sección reemplaza su `NotImplementedError`; cambios a este contrato se avisan en el issue.

In [ ]:
# Contrato compartido: nombres, firmas y convenciones que usa todo el notebook.
# Cada sección implementa su parte; nadie cambia esto sin avisar en el issue correspondiente.
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap

# --- Grilla y geometría -------------------------------------------------------
BASIS = "sto-3g"
D = np.round(np.concatenate([np.arange(0.6, 1.75, 0.1), np.arange(1.8, 3.01, 0.2)]), 2)  # Å, 19 puntos
STRETCHED = 1.8    # Å: desde acá hay que ganarle a CCSD
DISSOCIATED = 3.0  # Å: D_e = E(3,0) - E(R_e)
CHEM_ACC = 1.6e-3  # Ha


def h4_geometry(d: float) -> str:
    """Cadena lineal de 4 H sobre el eje z, separados por d Å (formato PySCF)."""
    return "; ".join(f"H 0 0 {i * d:.6f}" for i in range(4))


# --- Convención de qubits (Jordan-Wigner) --------------------------------------
# Orbitales espaciales MO de RHF, p = 0..3 en orden de energía.
# qubit p     = espín-orbital (p, α)
# qubit 4 + p = espín-orbital (p, β)
# |1> = ocupado. Qiskit es little-endian: el label de un estado se lee q7 ... q0.
N_ORB, N_QUBITS, N_ALPHA, N_BETA = 4, 8, 2, 2
HF_BITSTRING = "00110011"  # q0, q1 (α 0,1) y q4, q5 (β 0,1) ocupados


def alpha_qubit(p: int) -> int:
    return p


def beta_qubit(p: int) -> int:
    return N_ORB + p


# --- Transpilación y conteo de cz (igual para todos) ---------------------------
BASIS_GATES = ["cz", "sx", "x", "rz"]


def transpile_line(qc: QuantumCircuit) -> QuantumCircuit:
    return transpile(qc, basis_gates=BASIS_GATES, coupling_map=CouplingMap.from_line(qc.num_qubits),
                     optimization_level=3, seed_transpiler=0)


def cz_count(qc: QuantumCircuit) -> int:
    return transpile_line(qc).count_ops().get("cz", 0)


# --- Firmas que implementa cada sección ----------------------------------------
# Tablas compartidas (energías totales en Ha, con repulsión nuclear):
#   refs   : DataFrame  d, e_hf, e_ccsd, e_fci           (#1)
#   params : dict       d -> np.ndarray θ*               (#5)
#   vqe    : DataFrame  d, energy, sigma, shots, cz      (#6)

def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'}."""
    raise NotImplementedError


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    """#2. (H electrónico como SparsePauliOp de 8 qubits en la convención de arriba, e_nuc)."""
    raise NotImplementedError


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito parametrizado de 8 qubits que arranca en HF_BITSTRING."""
    raise NotImplementedError


## 1. Referencias clásicas (HF, CCSD, FCI)

Energías de referencia en toda la grilla `D` con PySCF. **FCI** es la respuesta exacta en la base STO-3G, contra la que se mide todo. **CCSD** es el método clásico a superar en la zona estirada. Ninguna de las dos es un resultado del VQE: solo sirven para comparar y para elegir puntos de partida.

Esta celda define `references(d)` y la tabla `refs` (`d, e_hf, e_ccsd, e_fci`), que también se guarda en `results/classical.csv`. Usa el mismo procedimiento de RHF que `h4_scf` de la sección 2 (`conv_tol = 1e-11`). Las energías no dependen del signo de los orbitales. Para cualquier cosa que dependa de los orbitales (vector CI, amplitudes), usar `h4_scf(d)`.

**Decisiones**

- Se usa **RHF**, no UHF. Desde ~1,2 Å RHF tiene una inestabilidad RHF → UHF: la solución UHF baja la energía pero rompe la simetría de espín. Queremos un singlete con orbitales restringidos, que es lo que usa el Hamiltoniano de qubits. El chequeo de **estabilidad interna** (RHF → RHF) pasa en todos los puntos: no hay una solución RHF más baja que la encontrada.
- Tolerancias ajustadas (`1e-11` en SCF, `1e-10` en CCSD, `1e-12` en FCI). El código verifica la convergencia de RHF y CCSD en cada punto y que el estado FCI sea singlete (⟨S²⟩ = 0). Todos convergen con el `init_guess` por defecto, así que no hizo falta arrancar desde la densidad del punto vecino.

**Qué muestran los números**

- Cerca del equilibrio (0,6–1,1 Å) CCSD está a menos de 0,01 mHa de FCI.
- Desde ~1,1 Å CCSD **deja de ser variacional** y queda por debajo de FCI. El error es −9,1 mHa a 1,8 Å (el punto más exigente de la zona juzgada), llega a −30,6 mHa a 2,6 Å y vuelve a −23,0 mHa a 3,0 Å.
- Para ganarle a CCSD hace falta `|E_VQE − E_FCI| + σ < 9,1 mHa` a 1,8 Å y < 18 mHa desde 2,0 Å.

In [ ]:
# 1. Referencias clásicas: HF, CCSD y FCI en toda la grilla D (#1).
# Solo sirven para comparar y elegir puntos de partida; nunca son resultados.
# Las energías no dependen del signo de los orbitales: los orbitales "oficiales" (signo fijo,
# los del Hamiltoniano) son los de h4_scf(d) en la sección 2. Acá se usa el mismo procedimiento.
import os
import matplotlib.pyplot as plt
from pyscf import cc, fci, gto, scf


def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'} en Ha (energías totales)."""
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    # Al estirar, el SCF podría caer en un punto silla. Si la solución tiene una inestabilidad
    # interna (RHF -> RHF), seguimos la dirección inestable y reconvergemos (en esta grilla no pasa).
    for _ in range(5):
        mo, _, stable, _ = mf.stability(return_status=True)
        if stable:
            break
        mf.kernel(dm0=mf.make_rdm1(mo, mf.mo_occ))
    assert mf.converged, f"RHF no convergió en d={d}"

    ccsd = cc.CCSD(mf)
    ccsd.conv_tol, ccsd.max_cycle = 1e-10, 500
    ccsd.kernel()
    assert ccsd.converged, f"CCSD no convergió en d={d}"

    solver = fci.FCI(mf)
    solver.conv_tol = 1e-12
    e_fci, ci = solver.kernel()
    s2, _ = solver.spin_square(ci, N_ORB, (N_ALPHA, N_BETA))
    assert abs(s2) < 1e-6, f"FCI no es singlete en d={d}: <S²>={s2:.3g}"
    return {"d": float(d), "e_hf": mf.e_tot, "e_ccsd": ccsd.e_tot, "e_fci": e_fci}


refs = pd.DataFrame([references(d) for d in D])

# Chequeos de cordura: FCI es exacto en la base (variacional), CCSD es casi exacto cerca del equilibrio.
assert (refs.e_fci <= refs.e_hf + 1e-10).all()
near_eq = refs[(refs.d >= 0.8) & (refs.d <= 1.0)]
assert ((near_eq.e_ccsd - near_eq.e_fci).abs() < CHEM_ACC).all()

os.makedirs("results", exist_ok=True)
refs.to_csv("results/classical.csv", index=False)

err_ccsd = 1e3 * (refs.e_ccsd - refs.e_fci)  # mHa, con signo
print(refs.assign(ccsd_menos_fci_mHa=err_ccsd).to_string(
    index=False, formatters={"d": "{:.2f}".format, "ccsd_menos_fci_mHa": "{:+.3f}".format}, float_format="%.6f"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.plot(refs.d, refs.e_hf, "s-", ms=4, label="RHF")
ax1.plot(refs.d, refs.e_ccsd, "^-", ms=4, label="CCSD")
ax1.plot(refs.d, refs.e_fci, "o-", ms=4, color="k", label="FCI (exacta)")
ax1.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0, label=f"zona estirada (d ≥ {STRETCHED} Å)")
ax1.set(xlabel="d (Å)", ylabel="E (Ha)", title="Curva de energía de H₄ (STO-3G)")
ax1.legend()

ax2.semilogy(refs.d, err_ccsd.abs(), "^-", ms=4, color="C1", label="|E_CCSD − E_FCI|")
ax2.axhline(1e3 * CHEM_ACC, ls="--", color="C2", label="precisión química (1,6 mHa)")
ax2.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0)
ax2.set(xlabel="d (Å)", ylabel="error (mHa)", title="Error de CCSD: lo que hay que superar")
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Hamiltoniano de qubits

In [ ]:
import copy
from functools import lru_cache

from pyscf import ao2mo, fci, gto, scf
from pyscf.fci import cistring


def _so(p: int, spin: int) -> int:
    return alpha_qubit(p) if spin == 0 else beta_qubit(p)


@lru_cache(maxsize=None)
def _jw_ladder(n_qubits: int) -> tuple[tuple[SparsePauliOp, ...], tuple[SparsePauliOp, ...]]:
    annihilation = []
    for j in range(n_qubits):
        z_string = list(range(j))
        annihilation.append(SparsePauliOp.from_sparse_list(
            [("Z" * j + "X", z_string + [j], 0.5), ("Z" * j + "Y", z_string + [j], 0.5j)],
            num_qubits=n_qubits))
    creation = [a.adjoint() for a in annihilation]
    return tuple(creation), tuple(annihilation)


def jordan_wigner(h1_so: np.ndarray, h2_so: np.ndarray | None = None, atol: float = 1e-12,
                  hermitian: bool = True) -> SparsePauliOp:
    n = h1_so.shape[0]
    cdag, a = _jw_ladder(n)
    terms = [SparsePauliOp("I" * n, 0.0)]
    for p, q in np.argwhere(np.abs(h1_so) > atol):
        terms.append(h1_so[p, q] * cdag[p] @ a[q])
    if h2_so is not None:
        for p, q, r, s in np.argwhere(np.abs(h2_so) > atol):
            if p == r or q == s:
                continue
            terms.append(0.5 * h2_so[p, q, r, s] * cdag[p] @ cdag[r] @ a[s] @ a[q])
    op = SparsePauliOp.sum(terms).simplify(atol=atol)
    if not hermitian:
        return op
    if np.max(np.abs(op.coeffs.imag), initial=0.0) > 1e-10:
        raise ValueError("Operador no hermítico (coeficientes complejos): revisar índices o usar hermitian=False")
    return SparsePauliOp(op.paulis, op.coeffs.real)


def spatial_to_spin_orbitals(h1: np.ndarray, eri: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n_orb = h1.shape[0]
    n_so = 2 * n_orb
    h1_so = np.zeros((n_so, n_so))
    h2_so = np.zeros((n_so, n_so, n_so, n_so))
    for s1 in (0, 1):
        i = [_so(p, s1) for p in range(n_orb)]
        h1_so[np.ix_(i, i)] = h1
        for s2 in (0, 1):
            k = [_so(p, s2) for p in range(n_orb)]
            h2_so[np.ix_(i, i, k, k)] = eri
    return h1_so, h2_so


def _distance_key(d: float) -> float:
    return round(float(d), 6)


@lru_cache(maxsize=None)
def _h4_scf_cached(d: float) -> scf.hf.RHF:
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    if not mf.converged:
        raise RuntimeError(f"RHF no convergió en d = {d}")
    c = mf.mo_coeff
    significant = np.abs(c) > 1e-4
    if not significant.any(axis=0).all():
        raise RuntimeError(f"Orbital sin coeficientes > 1e-4 en d = {d}: no se puede fijar el signo")
    first = np.argmax(significant, axis=0)
    mf.mo_coeff = c * np.sign(c[first, np.arange(c.shape[1])])
    return mf


def h4_scf(d: float) -> scf.hf.RHF:
    mf = copy.copy(_h4_scf_cached(_distance_key(d)))
    for attr in ("mo_coeff", "mo_energy", "mo_occ"):
        setattr(mf, attr, getattr(mf, attr).copy())
    return mf


@lru_cache(maxsize=None)
def _qubit_hamiltonian_cached(d: float) -> tuple[SparsePauliOp, float]:
    mf = _h4_scf_cached(d)
    mol, c = mf.mol, mf.mo_coeff
    h1 = c.T @ mf.get_hcore() @ c
    eri = ao2mo.restore(1, ao2mo.full(mol, c), c.shape[1])
    h1_so, h2_so = spatial_to_spin_orbitals(h1, eri)
    return jordan_wigner(h1_so, h2_so), mol.energy_nuc()


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    op, e_nuc = _qubit_hamiltonian_cached(_distance_key(d))
    return op.copy(), e_nuc


_STRINGS = cistring.make_strings(range(N_ORB), N_ALPHA)
SECTOR = np.array([int(a) | (int(b) << N_ORB) for a in _STRINGS for b in _STRINGS])


def sector_ground_energy(op: SparsePauliOp) -> float:
    m = op.to_matrix()
    return np.linalg.eigvalsh(m[np.ix_(SECTOR, SECTOR)])[0].real


def hf_energy_from_op(op: SparsePauliOp) -> float:
    i = int(HF_BITSTRING, 2)
    return op.to_matrix()[i, i].real


def validate_hamiltonian(d: float) -> dict:
    op, e_nuc = qubit_hamiltonian(d)
    mf = h4_scf(d)
    e_fci = fci.FCI(mf).kernel()[0]
    e_hf_op = hf_energy_from_op(op) + e_nuc
    e_gs_op = sector_ground_energy(op) + e_nuc
    return {"d": float(d), "n_paulis": len(op), "e_hf": mf.e_tot, "e_fci": e_fci,
            "err_hf": abs(e_hf_op - mf.e_tot), "err_fci": abs(e_gs_op - e_fci)}


ham_check = pd.DataFrame([validate_hamiltonian(d) for d in D])
assert (ham_check[["err_hf", "err_fci"]] < 1e-8).all().all(), ham_check
ham_check


### Decisiones (#2)

**Firma.** `qubit_hamiltonian(d) -> (op, e_nuc)`, como dice el contrato 0.1 ([ADR-0001](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0001-firma-qubit-hamiltonian.md)). `op` es solo la parte electrónica: energía total = $\langle op \rangle + e_\text{nuc}$. El estado HF es la constante `HF_BITSTRING = "00110011"` porque no depende de $d$.

**Orden de qubits.** Qubit $p$ = espín-orbital $(p, \alpha)$, qubit $4+p$ = $(p, \beta)$, con $p = 0..3$ los orbitales RHF ordenados por energía. `HF_BITSTRING` deja ocupados q0, q1, q4, q5 (Qiskit es little-endian: el label se lee q7 … q0).

**Hamiltoniano.** Integrales en base MO de RHF: $h_{pq} = C^T h^\text{core} C$ y $(pq|rs)$ con `ao2mo` (notación química). En espín-orbitales:

$$\hat H = \sum_{PQ} h_{PQ}\, a_P^\dagger a_Q + \tfrac12 \sum_{PQRS} (PQ|RS)\, a_P^\dagger a_R^\dagger a_S a_Q,$$

con el espín conservado dentro de cada par $(P,Q)$ y $(R,S)$. Jordan-Wigner a mano: $a_j = Z_0 \cdots Z_{j-1}\,(X_j + iY_j)/2$ ([ADR-0002](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0002-jordan-wigner-a-mano.md)). `jordan_wigner(h1_so, h2_so)` es genérica: sirve también para $N$, $S_z$, $S^2$ (#8) y para el tapering (#14). Por defecto exige un operador hermítico (coeficientes reales) y falla si no lo es, porque eso delata un error de índices; para operadores no hermíticos como $S^+ = \sum_p a^\dagger_{p\alpha} a_{p\beta}$ usar `hermitian=False`, que devuelve los coeficientes complejos.

**Signo de los orbitales.** PySCF puede devolver un orbital con el signo invertido entre una distancia y la siguiente. El espectro no cambia, pero los coeficientes de $\hat H$ sí, y eso rompe el *warm start* de #5. Se fija el signo para que el primer coeficiente de cada orbital con $|c| > 10^{-4}$ sea positivo (si algún orbital no tiene ninguno, se lanza un error en vez de anularlo) ([ADR-0003](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0003-signo-orbitales-y-cache.md)). Verificado: ningún término de Pauli cambia de signo de golpe entre distancias vecinas (los únicos cambios, cerca de 0,67 Å, son coeficientes que cruzan cero de forma suave).

**Orbitales RHF en toda la grilla.** RHF es inestable hacia UHF desde ~1,1 Å (`docs/analysis/state-of-the-art.md`; verificado con `mf.stability(external=True)`), pero se mantienen los orbitales RHF: solo definen la base (FCI no cambia), el estado de referencia sigue siendo un singlete y ⟨S²⟩ = 0 sigue sirviendo como diagnóstico en la sección 6 (#8). Con orbitales UHF la referencia quedaría contaminada de espín.

**Caché.** RHF y el Hamiltoniano se calculan una sola vez por distancia (`lru_cache` en funciones privadas). Las funciones públicas devuelven **copias**: `qubit_hamiltonian` devuelve `op.copy()` y `h4_scf` una copia del `mf` con sus propios `mo_coeff`, `mo_energy` y `mo_occ`, así que modificar lo devuelto (o correr `mf.kernel()` / `mf.stability()` sobre él) no altera el caché ni desincroniza orbitales y Hamiltoniano. La clave del caché es `d` redondeada a 6 decimales: `qubit_hamiltonian(0.1 * 7)` reutiliza el de `0.7`.

**Validación** (tabla `ham_check`, las 19 distancias):

| Prueba | Criterio | Resultado |
|---|---|---|
| $\langle \text{HF}\vert op \vert \text{HF}\rangle + e_\text{nuc}$ vs. $E_\text{HF}$ de PySCF | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Mínimo de `op` en el sector $N=4$, $S_z=0$ (36 estados) + $e_\text{nuc}$ vs. FCI | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Términos de Pauli | — | 185 (184 más la identidad) en todas las distancias, mismo conjunto de Paulis |

`SECTOR` son los 36 índices de la base computacional con 2 α y 2 β, **en el orden `ci[ia, ib]` de PySCF** (`cistring.make_strings`): un vector CI `ci` de PySCF corresponde a `psi[SECTOR] = ci.ravel()` (signos verificados contra `fci.direct_spin1.energy` en estados al azar, error < 3e-14 Ha). Es la misma definición que usa la sección 3 (#4).

La validación contra FCI se hace en el sector $N=4$, $S_z=0$: el mínimo del espectro completo (256 estados) puede tener otro número de electrones. FCI se calcula acá mismo con los orbitales de `h4_scf`; cuando esté `refs` (#1) se puede comparar también contra `refs.e_fci`.

**Qué no funcionó.** La primera versión armaba los productos con `A.compose(B)`, que en Qiskit es la matriz $B \cdot A$, no $A \cdot B$: quedaba $a_Q a_P^\dagger$ en vez de $a_P^\dagger a_Q$, o sea el Hamiltoniano con partículas y huecos intercambiados. Con 4 electrones en 8 espín-orbitales (medio llenado) ese operador tiene **el mismo espectro en el sector** $N=4$, así que la prueba contra FCI pasaba igual; solo la prueba de HF lo detectó (el estado HF aparecía en `11001100`). Se usa `@` (`A @ B` = $A \cdot B$). Por eso las dos pruebas son necesarias.

## 3. Ansatz

## 4. VQE

## 5. Resultados: curva, $R_e$, $D_e$ y tabla (energía, $\sigma$, shots, `cz`)

## 6. Zona estirada ($d \geq 1{,}8$ Å) vs. CCSD

## 7. Decisiones, qué no funcionó y limitaciones

## 8. Extras

### 8.1 Comparación de ansätze (precisión, profundidad y costo)

Comparamos cinco ansätze en **toda la grilla** (19 distancias). Todos arrancan en `HF_BITSTRING`:

| Ansatz | Qué es | Conserva |
|---|---|---|
| **HEA (RY + cz, 4 capas)** | Hardware-efficient tipo Kandala et al. (2017): capas de `ry` y una cadena de `cz` en línea | nada (ni N, ni S_z, ni S²) |
| **UCCSD** | Coupled cluster unitario, un paso de Trotter: dobles y después simples (occ → virt) | N, S_z |
| **k-UpCCGSD** (k = 1, 2, 3) | Lee et al. (2019): por capa, dobles de pares `(pα, pβ) → (qα, qβ)` y simples generalizadas con el mismo θ para α y β | N, S_z, S² |

**Cómo se mide cada cosa**

- **Parámetros (vale statevector):** simulador numpy propio con gradiente analítico (método adjunto) y L-BFGS-B. Los ansätze que conservan N y S_z se simulan en el sector de 36 estados en lugar de 256. Se barre la grilla con warm start en las dos direcciones, con 4 arranques en cada extremo (θ = 0 y 3 al azar), y se queda la menor energía por punto.
- **Control cruzado:** para cada punto, el circuito de Qiskit **transpilado** da la misma energía que el simulador numpy (`assert` a 1e-7 Ha).
- **`cz` y profundidad:** con `transpile_line` de la celda 0.1 (base `cz, sx, x, rz`, conectividad en línea, `optimization_level=3`). Las excitaciones fermiónicas se compilan como `PauliEvolutionGate` de sus cadenas de Jordan-Wigner.
- **Error medido:** `SamplerV2` de `qiskit_aer` con el **mismo presupuesto para todos**: 2²⁰ shots por punto, repartidos entre los grupos QWC en proporción a la norma 1 de sus coeficientes. E y σ salen de los bitstrings.
- **⟨S²⟩:** del estado final. Detecta contaminación con el triplete, que a 3,0 Å está a 0,46 mHa del fundamental.
- **Con ruido:** si #11 define `noise_model` (de `ruido.py`), se mide también con ruido en `d ≥ 1,8 Å` (matriz densidad, 2¹⁸ shots por punto).

El Hamiltoniano es `qubit_hamiltonian(d)` de la sección 2 (#2), y los operadores de escalera son los mismos (`_jw_ladder`).

Tarda ~7 min en Colab (~10 min más con ruido). Se puede saltear con `RUN_ANSATZ_COMPARISON = False`.

In [ ]:
# Extras · Comparación de ansätze (#15)
# Para cada familia y cada d de la grilla: error vs. FCI (statevector y medido con SamplerV2),
# cz y profundidad tras transpilar en línea, parámetros, evaluaciones del optimizador y <S²>.
# Si existe `noise_model` (#11, ruido.py), también el error medido con ruido.
RUN_ANSATZ_COMPARISON = True  # ~7 min en Colab; False para saltear esta celda
NOISY_SHOTS_PER_POINT = 2 ** 18
import itertools
import time
from functools import reduce

import scipy.sparse as sp
from qiskit.circuit import Parameter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Statevector
from qiskit_aer.primitives import SamplerV2
from scipy.optimize import minimize

# --- Operadores fermiónicos en Jordan-Wigner -------------------------------------
# Los de la sección 2 (#2): a_j = Z_0 ... Z_{j-1} (X_j + iY_j)/2, qubit p = (p, α), qubit 4 + p = (p, β).
_AD, _A = _jw_ladder(N_QUBITS)


def _s2_operator() -> SparsePauliOp:
    """S² = S_z² + S_z + S₋S₊ (en JW, con la convención de la celda 0.1)."""
    n = lambda q: _AD[q] @ _A[q]
    sz = 0.5 * SparsePauliOp.sum([n(alpha_qubit(p)) - n(beta_qubit(p)) for p in range(N_ORB)])
    s_plus = SparsePauliOp.sum([_AD[alpha_qubit(p)] @ _A[beta_qubit(p)] for p in range(N_ORB)])
    return (sz @ sz + sz + s_plus.adjoint() @ s_plus).simplify(atol=1e-12)


S2_OP = _s2_operator()

# --- Ansätze --------------------------------------------------------------------
# Cada ansatz es HF + una lista de factores exp(θ_k G_k), con G_k antihermítico y real.
#  - "exc": excitación fermiónica G = T − T†. Cumple G³ = −G => exp(θG) = I + sinθ G + (1 − cosθ) G².
#  - "rot": rotación de Pauli G = −iP/2   =>  exp(θG) = cos(θ/2) I + sin(θ/2) (−iP).
# Varios factores pueden compartir parámetro (k-UpCCGSD empareja α y β).


def _excitation(creates: list[int], annihilates: list[int]) -> SparsePauliOp:
    """Generador hermítico i(T − T†) con T = a†_c1 a†_c2 ... a_a2 a_a1."""
    T = reduce(lambda x, y: x @ y, [_AD[c] for c in creates] + [_A[a] for a in reversed(annihilates)])
    return (1j * (T - T.adjoint())).simplify(atol=1e-12)


def _single(p: int, q: int, s: int) -> SparsePauliOp:  # (q, s) -> (p, s)
    return _excitation([_so(p, s)], [_so(q, s)])


def _double(p: int, q: int, r: int, s_: int, s1: int, s2: int) -> SparsePauliOp:  # (r s1, s_ s2) -> (p s1, q s2)
    return _excitation([_so(p, s1), _so(q, s2)], [_so(r, s1), _so(s_, s2)])


OCC, VIRT = range(N_ALPHA), range(N_ALPHA, N_ORB)


def ansatz_uccsd():
    """UCCSD de un paso de Trotter (occ -> virt, conservan S_z). Orden: dobles y después simples."""
    ops = []
    for s in (0, 1):  # dobles con el mismo espín
        ops += [_double(a, b, i, j, s, s) for i, j in itertools.combinations(OCC, 2)
                for a, b in itertools.combinations(VIRT, 2)]
    ops += [_double(a, b, i, j, 0, 1) for i in OCC for j in OCC for a in VIRT for b in VIRT]  # αβ
    ops += [_single(a, i, s) for s in (0, 1) for i in OCC for a in VIRT]
    return [("exc", g, k) for k, g in enumerate(ops)], len(ops)


def ansatz_kupccgsd(k: int):
    """k-UpCCGSD (Lee et al. 2019): por capa, dobles de pares y después simples generalizadas
    (θ compartido entre α y β, así el ansatz conserva S²)."""
    gens, n = [], 0
    for _ in range(k):
        for p, q in itertools.combinations(range(N_ORB), 2):  # par (p α, p β) -> (q α, q β)
            gens.append(("exc", _double(q, q, p, p, 0, 1), n))
            n += 1
        for p, q in itertools.combinations(range(N_ORB), 2):
            gens += [("exc", _single(q, p, 0), n), ("exc", _single(q, p, 1), n)]
            n += 1
    return gens, n


def ansatz_hea(layers: int):
    """Hardware-efficient (Kandala et al. 2017, versión real): capas de RY + cadena de cz en línea."""
    gens, n = [], 0
    for layer in range(layers + 1):
        for q in range(N_QUBITS):
            gens.append(("rot", SparsePauliOp.from_sparse_list([("Y", [q], 1.0)], N_QUBITS), n))
            n += 1
        if layer < layers:
            gens.append(("cz", None, None))
    return gens, n


ANSATZ_FACTORIES = {
    "HEA (RY+cz, 4 capas)": lambda: ansatz_hea(4),
    "UCCSD": ansatz_uccsd,
    "1-UpCCGSD": lambda: ansatz_kupccgsd(1),
    "2-UpCCGSD": lambda: ansatz_kupccgsd(2),
    "3-UpCCGSD": lambda: ansatz_kupccgsd(3),
}

# --- Circuito de Qiskit equivalente (para contar cz y para medir) ---------------


def ansatz_circuit(gens, n_params: int) -> tuple[QuantumCircuit, list[Parameter]]:
    theta = [Parameter(f"θ{k:02d}") for k in range(n_params)]
    qc = QuantumCircuit(N_QUBITS)
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for kind, gen, k in gens:
        if kind == "cz":
            for q in range(N_QUBITS - 1):
                qc.cz(q, q + 1)
        elif kind == "rot":
            qc.ry(theta[k], int(np.flatnonzero(gen.paulis.x[0])[0]))  # exp(−iθY/2)
        else:  # exp(θ G) = exp(−i θ (iG)), y gen ya es iG (hermítico)
            qc.append(PauliEvolutionGate(gen, time=theta[k]), range(N_QUBITS))
    return qc, theta


# --- Simulador numpy con gradiente adjunto (solo para buscar parámetros) -------
_CZ_DIAG = np.array([(-1) ** sum((b >> q) & (b >> (q + 1)) & 1 for q in range(N_QUBITS - 1))
                     for b in range(2 ** N_QUBITS)], dtype=float)


def _real_generator(kind, gen):
    """Matriz real y rala del generador antihermítico G."""
    M = gen.to_matrix(sparse=True)  # = iG (exc) o P (rot)
    G = (-1j * M) if kind == "exc" else (-0.5j * M)
    assert abs(G.imag).max() < 1e-12
    return sp.csr_matrix(G.real)


# SECTOR (sección 2): los 36 estados con N = 4, S_z = 0. Acá solo se usa como conjunto de índices.


class NumpyAnsatz:
    """Simulador del ansatz. Si todos los factores conservan N y S_z, trabaja en el sector de 36 estados."""

    def __init__(self, gens, n_params):
        self.n = n_params
        self.idx = SECTOR if all(kind == "exc" for kind, _, _ in gens) else np.arange(2 ** N_QUBITS)
        self.ops = [(kind, None if kind == "cz" else self.proj(_real_generator(kind, gen)), k)
                    for kind, gen, k in gens]
        self.hf = (self.idx == int(HF_BITSTRING, 2)).astype(float)

    def proj(self, M):
        return sp.csr_matrix(M[self.idx][:, self.idx])

    @staticmethod
    def _apply(kind, G, t, v):
        if kind == "cz":  # solo en el espacio completo (HEA)
            return _CZ_DIAG * v
        Gv = G @ v
        if kind == "exc":
            return v + np.sin(t) * Gv + (1 - np.cos(t)) * (G @ Gv)
        return np.cos(t / 2) * v + 2 * np.sin(t / 2) * Gv

    def state(self, theta):
        v = self.hf
        for kind, G, k in self.ops:
            v = self._apply(kind, G, None if k is None else theta[k], v)
        return v

    def energy_and_grad(self, theta, H):
        psi = self.state(theta)
        lam = H @ psi
        e = psi @ lam
        grad = np.zeros(self.n)
        for kind, G, k in reversed(self.ops):  # dE/dθ = 2 <λ_k| G |ψ_k>
            if kind != "cz":
                grad[k] += 2 * lam @ (G @ psi)
            t = None if k is None else -theta[k]
            psi, lam = self._apply(kind, G, t, psi), self._apply(kind, G, t, lam)
        return e, grad


def _minimize(model, H, x0):
    res = minimize(model.energy_and_grad, x0, args=(H,), jac=True, method="L-BFGS-B",
                   options={"maxiter": 5000, "ftol": 1e-14, "gtol": 1e-9})
    return res.x, res.fun, res.nfev


def optimize_grid(model, Hs, n_random=3, seed=0):
    """Barrido con warm start en las dos direcciones + multi-start en los extremos; por punto, el mejor."""
    rng = np.random.default_rng(seed)
    best = {}
    Hs = {d: model.proj(H) for d, H in Hs.items()}

    def keep(d, x, e, nfev):
        if d not in best or e < best[d][1] - 1e-12:
            best[d] = (x, e, nfev)

    def starts():
        return [np.zeros(model.n)] + [rng.normal(0, 0.1, model.n) for _ in range(n_random)]

    for order in (list(D), list(D)[::-1]):
        x = min((_minimize(model, Hs[order[0]], x0) for x0 in starts()), key=lambda r: r[1])[0]
        for d in order:
            x, e, nfev = _minimize(model, Hs[d], x)
            keep(d, x, e, nfev)
    return best


# --- Medición con SamplerV2 (grupos QWC, shots ∝ peso del grupo) -----------------
SHOTS_PER_POINT = 2 ** 20  # mismo presupuesto por punto para todos los ansätze
# Ojo: con SamplerV2(seed=s), Aer repite semillas entre pubs y las mediciones de distintos grupos
# quedan correlacionadas (la σ calculada subestima el error real). Una semilla independiente por pub lo evita.
_seeds = np.random.default_rng(1234)


def _rotate_to_z(qc: QuantumCircuit, basis_label: str) -> QuantumCircuit:
    m = qc.copy()
    for q, b in enumerate(reversed(basis_label)):
        if b == "X":
            m.h(q)
        elif b == "Y":
            m.sdg(q)
            m.h(q)
    m.measure_all()
    return transpile(m, basis_gates=BASIS_GATES, optimization_level=0)


def measure_energies(tqc, theta, xs, ops_l, shots_per_point=SHOTS_PER_POINT, noise_model=None):
    """E y σ medidos para cada fila de xs (un juego de parámetros por d), sobre el circuito transpilado.

    ops_l: SparsePauliOp ya con el layout del circuito transpilado, uno por d (mismo orden que xs).
    """
    labels = sorted(set().union(*[set(op.paulis.to_labels()) for op in ops_l]) - {"I" * N_QUBITS})
    groups = SparsePauliOp(labels).group_commuting(qubit_wise=True)
    coeff = [dict(zip(op.paulis.to_labels(), op.coeffs.real)) for op in ops_l]
    weight = np.array([np.mean([sum(abs(c.get(l, 0.0)) for l in g.paulis.to_labels()) for c in coeff]) for g in groups])
    shots = np.maximum(100, np.round(shots_per_point * weight / weight.sum())).astype(int)

    pubs = []
    for g, n in zip(groups, shots):
        basis = "".join(max(col, key=lambda ch: ch != "I") for col in zip(*g.paulis.to_labels()))
        circ = _rotate_to_z(tqc, basis)
        pubs.append((circ, np.array([[x[theta.index(p)] for p in circ.parameters] for x in xs]), int(n)))
    options = {} if noise_model is None else {
        "backend_options": {"method": "density_matrix", "noise_model": noise_model}}
    results = [SamplerV2(seed=int(_seeds.integers(2 ** 31)), options=options).run([pub]).result()[0]
               for pub in pubs]

    bits = np.array([[(b >> q) & 1 for q in range(N_QUBITS)] for b in range(2 ** N_QUBITS)])
    E = np.array([c.get("I" * N_QUBITS, 0.0) for c in coeff])
    var = np.zeros(len(xs))
    for g, res in zip(groups, results):
        ls = g.paulis.to_labels()
        masks = np.array([[ch != "I" for ch in reversed(l)] for l in ls])          # (términos, qubits)
        signs = (-1.0) ** (bits @ masks.T.astype(int))                              # (256, términos)
        for j in range(len(xs)):
            v = signs @ np.array([coeff[j].get(l, 0.0) for l in ls])               # valor por shot, por bitstring
            cnt = res.data.meas.get_int_counts(loc=j)
            idx, n_b = np.fromiter(cnt.keys(), int), np.fromiter(cnt.values(), float)
            n = n_b.sum()
            mean = n_b @ v[idx] / n
            E[j] += mean
            var[j] += (n_b @ (v[idx] - mean) ** 2) / (n - 1) / n
    return E, np.sqrt(var), int(shots.sum())


# --- Corrida -----------------------------------------------------------------
if RUN_ANSATZ_COMPARISON:
    t0 = time.time()
    NOISE_MODEL = globals().get("noise_model")  # lo define #11 a partir de ruido.py
    H_OPS = {d: qubit_hamiltonian(d) for d in D}
    H_MATS = {d: sp.csr_matrix(op.to_matrix(sparse=True).real) for d, (op, _) in H_OPS.items()}
    S2_MAT = sp.csr_matrix(S2_OP.to_matrix(sparse=True).real)
    e_fci = refs.set_index("d").e_fci

    rows, ansatz_summary = [], []
    for name, factory in ANSATZ_FACTORIES.items():
        t = time.time()
        gens, n_params = factory()
        model = NumpyAnsatz(gens, n_params)
        best = optimize_grid(model, H_MATS)

        qc, theta = ansatz_circuit(gens, n_params)
        tqc = transpile_line(qc)
        ops_l = [H_OPS[d][0].apply_layout(tqc.layout) for d in D]
        xs = [best[d][0] for d in D]

        # Control cruzado: el circuito de Qiskit transpilado da la misma energía que el simulador numpy.
        for d, x, op_l in zip(D, xs, ops_l):
            e_qiskit = Statevector(tqc.assign_parameters(dict(zip(theta, x)))).expectation_value(op_l).real
            assert abs(e_qiskit - best[d][1]) < 1e-7, f"{name}: circuito ≠ modelo en d={d}"

        e_meas, sigma, shots = measure_energies(tqc, theta, xs, ops_l)
        noisy = {}
        if NOISE_MODEL is not None:  # con ruido: solo la zona estirada (matriz densidad, es más lento)
            js = [j for j, d in enumerate(D) if d >= STRETCHED]
            e_n, s_n, _ = measure_energies(tqc, theta, [xs[j] for j in js], [ops_l[j] for j in js],
                                           NOISY_SHOTS_PER_POINT, NOISE_MODEL)
            noisy = {D[j]: (e_n[i], s_n[i]) for i, j in enumerate(js)}
        for j, d in enumerate(D):
            x, e_sv, nfev = best[d]
            psi, s2_mat = model.state(x), model.proj(S2_MAT)
            e_nuc = H_OPS[d][1]
            rows.append({"ansatz": name, "d": d,
                         "err_sv_mHa": 1e3 * (e_sv + e_nuc - e_fci[d]),
                         "err_meas_mHa": 1e3 * (e_meas[j] + e_nuc - e_fci[d]),
                         "sigma_mHa": 1e3 * sigma[j], "s2": psi @ (s2_mat @ psi), "nfev": nfev,
                         "err_noisy_mHa": 1e3 * (noisy[d][0] + e_nuc - e_fci[d]) if d in noisy else np.nan,
                         "sigma_noisy_mHa": 1e3 * noisy[d][1] if d in noisy else np.nan})
        ops = tqc.count_ops()
        ansatz_summary.append({"ansatz": name, "params": n_params, "cz": ops.get("cz", 0), "depth": tqc.depth(),
                        "shots/punto": shots, "t (s)": round(time.time() - t, 1)})
        print(f"{name:22s} listo en {time.time() - t:5.1f} s")

    ansatz_cmp = pd.DataFrame(rows)
    ansatz_cmp.to_csv("results/ansatz_comparison.csv", index=False)
    stretched = ansatz_cmp[ansatz_cmp.d >= STRETCHED]
    ansatz_summary = pd.DataFrame(ansatz_summary).set_index("ansatz")
    ansatz_summary["max err sv (d ≥ 1,8) mHa"] = stretched.groupby("ansatz").err_sv_mHa.max()
    ansatz_summary["max err sv (toda) mHa"] = ansatz_cmp.groupby("ansatz").err_sv_mHa.max()
    ansatz_summary["max |err med|+σ (d ≥ 1,8) mHa"] = (stretched.err_meas_mHa.abs() + stretched.sigma_mHa).groupby(stretched.ansatz).max()
    if NOISE_MODEL is not None:
        ansatz_summary["max |err ruido| (d ≥ 1,8) mHa"] = stretched.err_noisy_mHa.abs().groupby(stretched.ansatz).max()
    ansatz_summary["max <S²>"] = ansatz_cmp.groupby("ansatz").s2.max()
    ansatz_summary["nfev medio"] = ansatz_cmp.groupby("ansatz").nfev.mean().round(0)
    ansatz_summary["le gana a CCSD"] = stretched.assign(
        ok=lambda df: df.err_meas_mHa.abs() + df.sigma_mHa < 1e3 * (refs.set_index("d").e_ccsd - e_fci).abs()[df.d].values
    ).groupby("ansatz").ok.all()
    print(f"Total: {time.time() - t0:.0f} s")
    print(ansatz_summary.to_string(float_format=lambda v: f"{v:.3g}"))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    for name, df in ansatz_cmp.groupby("ansatz", sort=False):
        ax1.semilogy(df.d, df.err_sv_mHa.clip(lower=1e-6), "o-", ms=3, label=f"{name} ({ansatz_summary.loc[name, 'cz']} cz)")
        ax2.errorbar(df.d, df.err_meas_mHa, yerr=df.sigma_mHa, fmt="o-", ms=3, capsize=2, label=name)
    ccsd_err = 1e3 * (refs.e_ccsd - refs.e_fci).abs()
    ax1.semilogy(refs.d, ccsd_err, "k--", label="|CCSD − FCI|")
    for ax in (ax1, ax2):
        ax.axhline(1e3 * CHEM_ACC, color="C2", ls=":", label="precisión química")
        ax.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0)
        ax.set_xlabel("d (Å)")
    ax1.set(ylabel="E_VQE − E_FCI (mHa)", title="Error con statevector (piso en 1e-6 mHa = exacto)")
    ax2.plot(refs.d, ccsd_err, "k--")
    ax2.set(ylabel="E_medida − E_FCI (mHa)", title=f"Error medido con SamplerV2 ({SHOTS_PER_POINT:.0e} shots/punto)")
    ax1.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

#### Resultados

| Ansatz | Parámetros | `cz` | Profundidad | Máx. error sv, d ≥ 1,8 (mHa) | Máx. error sv, toda la curva (mHa) | Máx. \|error medido\| + σ, d ≥ 1,8 (mHa) | Máx. ⟨S²⟩ | ¿Le gana a CCSD? |
|---|---:|---:|---:|---:|---:|---:|---:|:---:|
| HEA (4 capas) | 40 | 28 | 35 | 44,1 | 98,4 | 44–48 | 1,99 | no |
| UCCSD | 26 | 1303 | 4220 | 3,5 | 3,5 | 8,6 | 0,1 | sí (justo) |
| 1-UpCCGSD | 12 | 474 | 1451 | 11,5 | 16,9 | 17,2 | 0 | no |
| **2-UpCCGSD** | 24 | 982 | 2915 | **< 1e-9** | **< 1e-9** | 6,0 | 0 | **sí** |
| 3-UpCCGSD | 36 | 1490 | 4379 | < 1e-9 | < 1e-9 | 6,8 | 0 | sí |

Los números del HEA varían un poco entre corridas: su optimización tiene muchos mínimos locales y el álgebra lineal multihilo no es determinista.

#### Conclusión

- **2-UpCCGSD** es el que conviene si no hay ruido. Llega a FCI en toda la curva con solo 24 parámetros y conserva S² por construcción: no se contamina con el triplete casi degenerado de la zona estirada. 3-UpCCGSD no agrega nada y suma 50 % más de `cz`.
- **UCCSD** se traba en 3,5 mHa entre 1,8 y 2,2 Å. Le gana a CCSD por poco a 1,8 Å (8,6 contra 9,1 mHa) y nunca llega a precisión química en esa zona. **1-UpCCGSD** no alcanza.
- El **HEA** es 35 veces más barato en `cz`, pero se queda en mínimos locales (44 mHa a 1,8 Å) y rompe la simetría de espín (⟨S²⟩ ≈ 2): energía baja no significa estado correcto.
- **Lo que limita al ansatz exacto es la medición**, no el ansatz. Con 2²⁰ shots por punto, σ ≈ 2,3 mHa. Para tener precisión química medida hacen falta ~8 veces más shots (σ ≈ 0,8 mHa) o una medición más eficiente (#6).
- **Con ruido, la elección se invierte.** En una prueba con un modelo sintético igual al del enunciado (depolarizante, 0,1 % por `cz` y 0,005 % por `sx`/`x`), 2-UpCCGSD queda a ~450 mHa de FCI y el HEA a 21–67 mHa. Con ~1000 `cz`, el error por compuerta lo domina todo. Reducir `cz` (excitaciones compiladas como compuertas de Givens o QEB en lugar de cadenas de Pauli, #4) es lo que hace falta para que el ansatz exacto sirva con ruido.

#### Qué no funcionó

- **El orden de los factores importa mucho.** Con simples antes que dobles, 1-UpCCGSD llega a 220 mHa de error a 3,0 Å (con el orden inverso, a 0,3 mHa). UCCSD llega a 5,2 mHa y se contamina con el triplete (⟨S²⟩ hasta 1,9). Por eso todos aplican primero las dobles y después las simples, que actúan como una rotación orbital final.
- **Más capas de HEA no ayudan.** Con 8 capas (72 parámetros) da exactamente el mismo error que con 4 y tarda el doble: es un problema de optimización (mínimos locales), no de expresividad. Lo sacamos de la comparación.
- **`SamplerV2(seed=...)` de Aer repite semillas entre pubs.** Las mediciones de distintos grupos quedaban correlacionadas, la σ subestimaba el error real y aparecían desvíos de 3–4σ. Con una semilla independiente por pub, los z-scores de los ansätze exactos tienen desvío 1,06 en 40 puntos (máx. |z| = 2,8): la σ está bien calibrada. **Ojo #6:** les pasa lo mismo si usan una semilla fija.